# Contract-quality pilot: saved 1_5b adapter
Exploratory synthetic **draft**, open development only. No training and no Stage 2 generation/repair.
Select **GPU T4 x2**, Internet on. Attach a private Kaggle dataset containing the original adapter folders
`qwen25_1_5b/adapter/` and `qwen25_3b/adapter/` (all files). Automatic discovery requires exactly one matching adapter.
The notebook embeds its code and suite: **no GitHub push is needed**. A CPU-validated notebook is not a GPU execution guarantee.
A four-call smoke run precedes the 160-call paired experiment. Do not change the method after viewing selection labels.

In [ ]:
import base64, hashlib, json, os, subprocess, sys, zlib
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
SIZE='1_5b'
PAYLOAD_SHA='97d6a1f459e1b07cb5a7f9eeb56de3a69135d2ded821b1638658dcfe45f98825'
PAYLOAD=(
'...'
'4b8BYwn7CA==')
raw=zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA
ROOT=WORK/('cq_source_'+PAYLOAD_SHA[:12])
for name,source in json.loads(raw).items():
    relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():assert path.read_text()==source, f'Changed embedded file: {path}'
    else:path.write_text(source)
spec=json.loads((ROOT/'benchmark/contract_quality_v1/adapters.json').read_text())[SIZE]
matches=[]
for path in Path('/kaggle/input').rglob('adapter_config.json'):
    try:
        if json.loads(path.read_text())['base_model_name_or_path']==spec['model']:
            matches.append(path.parent)
    except (KeyError,ValueError):pass
assert len(matches)==1, f'Expected one adapter for {SIZE}; found {matches}. Attach the original adapter dataset.'
ADAPTER=matches[0]
for name,expected in spec['sha256'].items():
    assert hashlib.sha256((ADAPTER/name).read_bytes()).hexdigest()==expected, f'Wrong adapter file: {name}'
print('Payload:', PAYLOAD_SHA, 'Adapter:', ADAPTER)


In [ ]:
# Install CPU/verifier packages separately; use only child processes for heavy imports.
PACKAGES=WORK/'cq_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1')
probe=subprocess.run([sys.executable,'-c',"import torch; assert torch.__version__ == '2.11.0+cu130', f'Expected torch 2.11.0+cu130, found {torch.__version__}'; print(torch.__version__); assert torch.cuda.device_count()==2"],cwd=ROOT,env=ENV,check=True)
print('Environment ready. If the pinned torch check fails, stop and share the setup log; do not silently change runtime.')


In [ ]:
def run_pilot(smoke):
    name='cq_'+SIZE+('_smoke' if smoke else '_pilot')
    out=WORK/name
    command=[sys.executable,'-m','eval.contract_quality.pilot','--size',SIZE,'--adapter',str(ADAPTER),'--out',str(out)]
    if smoke:command.append('--smoke')
    log=WORK/(name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Run failed: {log}. Download the log and any partial ZIP; do not score an incomplete run.'
    done=json.loads((out/'completion.json').read_text())
    assert done['complete'] and done['records']==(4 if smoke else 160)
    print('Bundle:',str(out)+'.zip')
run_pilot(smoke=True)


In [ ]:
run_pilot(smoke=False)
from IPython.display import FileLink, display
display(FileLink(str(WORK/('cq_'+SIZE+'_pilot.zip'))))


Download `cq_*_pilot.zip` and the executed notebook. Put them in
`kaggle_outputs/contract_quality/v1/` and `notebooks/executed/contract_quality/v1/` respectively.
Run both adapter notebooks, then follow `docs/contract_quality/RUNBOOK.md` for local scoring.
The smoke ZIP is only an infrastructure check, not an experiment result.